# Orthogonal constraint steering — Kaggle runner (2× T4)Every story is saved to `state.json` the moment it is generated. If this notebookstops for any reason, **re-run the same cell** and it continues from where it leftoff. Nothing already generated is repeated.Accelerator must be set to **GPU T4 x2** (Settings → Accelerator).

In [ ]:
MODEL   = "Fanar"                        # Fanar | ALLaM | AceGPT | Jais | Phi-4-miniSUITE   = "core"                         # core | ortho | beta | loo | allSTORIES = 50                             # stories per conditionOUT     = "/kaggle/working/orthosteer"   # checkpoint + results live hereREPO    = "/kaggle/working/NoiseEGRA"BRANCH  = "NoiseSteering"

## 1. Install and clone

In [ ]:
!pip install -q -U "transformers>=4.56" accelerate hf_transfer!rm -rf {REPO} && git clone -q --branch {BRANCH} --single-branch \    https://github.com/haziq-exe/NoiseEGRA.git {REPO}import os# HF's default downloader is single-threaded and stalls on large shards.# hf_transfer is a multi-threaded Rust downloader; 16 GB lands in a few minutes.os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Hugging Face loginNeeded for gated models (Fanar and ALLaM are). Add your token under**Add-ons → Secrets** with the name `HF_TOKEN`. Skip this cell for ungated models.

In [ ]:
from kaggle_secrets import UserSecretsClientfrom huggingface_hub import loginlogin(UserSecretsClient().get_secret("HF_TOKEN"))

## 3. Continuing from an earlier session (skip on the first run)Kaggle wipes `/kaggle/working` when a session ends. To carry the checkpoint over:**Save Version** at the end of a session, then in the new session add that version'soutput under **Add-ons → Add data → Your Work**, put its path below, and run this cell.

In [ ]:
import os, shutilPREVIOUS = ""   # e.g. "/kaggle/input/orthosteer-run-1/orthosteer"if PREVIOUS and os.path.isdir(PREVIOUS):    shutil.copytree(PREVIOUS, OUT, dirs_exist_ok=True)    import json    n = sum(len(v) for v in json.load(open(f"{OUT}/state.json"))["runs"].values())    print(f"restored {n} stories from {PREVIOUS}")else:    print("starting fresh")

## 4. GenerateThe first run downloads the model weights (14-18 GB). With `hf_transfer` enabled abovethis takes a few minutes; without it, HF's single-threaded downloader can stall to acrawl near the end. If the progress bar drops below ~1 MB/s, interrupt and re-run thiscell — it restarts the download at full speed and no generated stories are lost.After that it prints one line per story with a running estimate of time remaining.Safe to interrupt at any point; re-run to resume.

In [ ]:
!cd {REPO} && python -u scripts/kaggle_orthosteer.py \    --model {MODEL} --suite {SUITE} --num-stories {STORIES} --out {OUT}

## 5. ScoreCounts how many stories obey each constraint. No judge model, no API calls.

In [ ]:
!cd {REPO} && python -u scripts/score_orthosteer.py --input-dir {OUT}

Results are in `/kaggle/working/orthosteer/`:| file | what it is ||---|---|| `state.json` | the checkpoint, and the archive of every story generated || `<condition>.csv` | one story per row || `EXACT_SCORES/Ortho_Constraint_Table.md` | the comparison table across conditions || `EXACT_SCORES/<condition>.csv` | per-story measurements |**Save Version** before closing so `/kaggle/working` is kept.